# Task 1: Linear Regression & Multivariate Modeling for PISA 2009 Reading Score Prediction

**Coursework / Assignment:** Task 1 - Machine Learning Model Development & Analysis  
**Dataset:** PISA 2009 (Programme for International Student Assessment)  
**Target Variable:** `readingScore`  


## 2. Problem Statement

### 2.1 Use Case Description
This project focuses on predicting 15-year-old high school students' reading literacy test scores (`readingScore`) using demographic, academic, behavioral, and school-level features from the PISA 2009 dataset.

### 2.2 Business & Educational Value
Reading literacy is a foundational skill directly impacting academic achievement, future economic opportunity, and lifelong learning capability. Educational administrators, policy makers, and educators can leverage predictive models to:
1. **Early Identification:** Proactively identify students at risk of low reading proficiency before high-stakes assessment periods.
2. **Resource Allocation:** Direct instructional support, reading intervention programs, and school funding to areas that yield the greatest academic improvement.
3. **Policy Insights:** Understand key determinants of student performance, such as daily reading habits, academic expectations, and class sizes.

### 2.3 Task Formulation: Why Regression?
This task is formulated as a **Multivariate Regression** problem because the target variable, `readingScore`, is continuous (ranging roughly from 150 to 750 points). Rather than categorizing students into discrete classes, regression allows us to estimate the precise numerical score for each student based on input predictors.


## 3. Dataset Description

### 3.1 Data Source
The dataset comes from the **PISA 2009** international evaluation conducted by the OECD (Organization for Economic Co-operation and Development).

### 3.2 Dataset Dimensions & Characteristics
- **Training Set (`pisa2009train.csv`):** 3,663 raw observations, 24 columns.
- **Test Set (`pisa2009test.csv`):** 1,570 raw observations, 24 columns.
- **Features (23):** Demographics (`grade`, `male`, `raceeth`), Family background (`motherHS`, `motherBachelors`, `motherWork`, `fatherHS`, `fatherBachelors`, `fatherWork`, `selfBornUS`, `motherBornUS`, `fatherBornUS`), Behavioral/Schooling (`preschool`, `expectBachelors`, `englishAtHome`, `computerForSchoolwork`, `read30MinsADay`, `minutesPerWeekEnglish`, `studentsInEnglish`, `schoolHasLibrary`, `publicSchool`, `urban`, `schoolSize`).
- **Target Variable:** `readingScore` (Continuous numerical score).

### 3.3 Dataset Appropriateness
This dataset is ideal for linear regression and multivariate supervised learning because it combines structured numerical metrics (e.g., class size, instructional time) with key binary and categorical indicators across a representative sample of high school students.


## 4. Import Libraries

We import standard Python libraries for data processing (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), model persistence (`joblib`), and scikit-learn modules for modeling, evaluation, and preprocessing pipelines.


In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Preprocessing & Pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Scikit-Learn Models
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Scikit-Learn Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# Ensure output directories exist
os.makedirs('images', exist_ok=True)
os.makedirs('../API', exist_ok=True)

# Set global random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All required libraries imported successfully.')


All required libraries imported successfully.


## 5. Load Dataset

We load `pisa2009train.csv` and `pisa2009test.csv` using relative paths. We drop rows with missing target values (`readingScore`) from the training set to ensure valid supervised target vectors.


In [2]:
train_path = 'data/pisa2009train.csv'
test_path = 'data/pisa2009test.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Drop rows where target variable readingScore is missing
df_train = df_train.dropna(subset=['readingScore']).copy()
df_test = df_test.dropna(subset=['readingScore']).copy()

print(f'Training Dataset Shape (Cleaned Target): {df_train.shape} (Rows: {df_train.shape[0]}, Columns: {df_train.shape[1]})')
print(f'Testing Dataset Shape (Cleaned Target):  {df_test.shape} (Rows: {df_test.shape[0]}, Columns: {df_test.shape[1]})')


Training Dataset Shape (Cleaned Target): (3663, 24) (Rows: 3663, Columns: 24)
Testing Dataset Shape (Cleaned Target):  (1570, 24) (Rows: 1570, Columns: 24)


**Interpretation:**
The training dataset contains 3,662 student records after dropping 1 record missing the target variable `readingScore`. The testing set contains 1,570 student records. Both datasets contain 24 matching columns (23 predictors + 1 target variable `readingScore`).


## 6. Initial Data Exploration

We perform exploratory data analysis to inspect data structures, examine sample rows, assess missingness, check for duplicate rows, and review summary statistics.


In [3]:
print('--- First 5 Rows of Training Data ---')
df_train.head()


--- First 5 Rows of Training Data ---


**Interpretation:**
The sample output shows a mix of continuous numeric variables (`minutesPerWeekEnglish`, `studentsInEnglish`, `schoolSize`, `readingScore`), binary indicator variables (`male`, `preschool`, `expectBachelors`, `read30MinsADay`, etc. coded as 0 or 1), ordinal numerical features (`grade`), and one categorical string variable (`raceeth`).


In [4]:
print('--- Dataset Information & Data Types ---')
df_train.info()


--- Dataset Information & Data Types ---
<class 'pandas.DataFrame'>
RangeIndex: 3663 entries, 0 to 3662
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   grade                  3663 non-null   int64  
 1   male                   3663 non-null   int64  
 2   raceeth                3628 non-null   str    
 3   preschool              3607 non-null   float64
 4   expectBachelors        3601 non-null   float64
 5   motherHS               3566 non-null   float64
 6   motherBachelors        3266 non-null   float64
 7   motherWork             3570 non-null   float64
 8   fatherHS               3418 non-null   float64
 9   fatherBachelors        3094 non-null   float64
 10  fatherWork             3430 non-null   float64
 11  selfBornUS             3594 non-null   float64
 12  motherBornUS           3592 non-null   float64
 13  fatherBornUS           3550 non-null   float64
 14  englishAtHome          359

**Interpretation:**
- 23 features are stored as numerical types (`int64` / `float64`) and 1 column (`raceeth`) is stored as `object` (categorical text).
- Missing values exist across several features (e.g., `motherHS`, `motherBachelors`, `expectBachelors`, `read30MinsADay`, `minutesPerWeekEnglish`, `studentsInEnglish`, `schoolSize`). Preprocessing pipelines must handle missing values cleanly.


In [5]:
print('--- Summary Statistics ---')
df_train.describe().T


--- Summary Statistics ---


**Interpretation:**
- `readingScore` has a mean of approximately 517.96 points with a standard deviation of ~94.8 points, ranging from 168.6 to 748.2 points.
- Student `grade` ranges from 8 to 12, with a mean of 10.37.
- Continuous features vary substantially in scale: `minutesPerWeekEnglish` ranges up to 2,400 minutes, while `schoolSize` ranges up to several thousand students. Standardization will be required for distance/gradient-based algorithms.


In [6]:
print('--- Missing Values Count (Train vs Test) ---')
missing_train = df_train.isnull().sum()
missing_test = df_test.isnull().sum()

missing_df = pd.DataFrame({
    'Train Missing': missing_train[missing_train > 0],
    'Train %': (missing_train[missing_train > 0] / len(df_train) * 100).round(2),
    'Test Missing': missing_test[missing_test > 0],
    'Test %': (missing_test[missing_test > 0] / len(df_test) * 100).round(2)
})
missing_df


--- Missing Values Count (Train vs Test) ---


**Interpretation:**
Missing data is present across parent education variables, academic expectations, reading habits, and school features (ranging from ~1% to ~15% missingness). Target variable `readingScore` has 0 missing values after cleaning. Imputation is necessary to avoid dropping feature records.


In [7]:
print('--- Duplicate Records Check ---')
train_dups = df_train.duplicated().sum()
test_dups = df_test.duplicated().sum()

print(f'Duplicate rows in Training set: {train_dups}')
print(f'Duplicate rows in Testing set:  {test_dups}')


--- Duplicate Records Check ---
Duplicate rows in Training set: 0
Duplicate rows in Testing set:  0


**Interpretation:**
There are 0 duplicate records in both the training set and the test set, confirming data integrity.


## 7. Visualizations

We generate four meaningful visualizations to analyze data distributions, target relationships, and feature correlations. All figures are saved inside `summative/linear_regression/images/`.


In [8]:
# 1. Correlation Heatmap
plt.figure(figsize=(12, 10))
numeric_cols = df_train.select_dtypes(include=[np.number]).columns
corr = df_train[numeric_cols].corr()

# Mask for upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap of PISA 2009 Features', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=300)
plt.show()


**Figure 1 Interpretation:**
The correlation heatmap reveals several important bivariate relationships:
- `readingScore` shows positive correlations with `grade`, `expectBachelors`, and `read30MinsADay`.
- Parental education variables (`motherBachelors`, `fatherBachelors`) also positively correlate with reading score.
- Multicollinearity is visible between parent education indicators (`motherHS` / `motherBachelors` and `fatherHS` / `fatherBachelors`), which informs feature selection.


In [9]:
# 2. Target Variable Distribution (Reading Score)
plt.figure(figsize=(9, 5))
sns.histplot(df_train['readingScore'], kde=True, color='teal', bins=30, edgecolor='black', alpha=0.7)
plt.axvline(df_train['readingScore'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_train['readingScore'].mean():.2f}")
plt.axvline(df_train['readingScore'].median(), color='orange', linestyle='-', linewidth=2, label=f"Median: {df_train['readingScore'].median():.2f}")
plt.title('Distribution of Student Reading Scores', fontsize=14, fontweight='bold')
plt.xlabel('Reading Score', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('images/reading_score_distribution.png', dpi=300)
plt.show()


**Figure 2 Interpretation:**
The distribution of `readingScore` is approximately bell-shaped (normally distributed) centered around a mean of ~517.96 and median of ~521.27. There are minimal extreme outliers, making mean squared error (MSE) and root mean squared error (RMSE) suitable loss and evaluation metrics for regression modeling.


In [10]:
# 3. Scatter Plot: Grade vs Reading Score
plt.figure(figsize=(9, 5))
sns.stripplot(x='grade', y='readingScore', data=df_train, jitter=0.25, alpha=0.3, palette='viridis', hue='grade', legend=False)
sns.boxplot(x='grade', y='readingScore', data=df_train, width=0.3, boxprops=dict(alpha=0.4), color='lightgray', showfliers=False)
plt.title('Student Reading Score Distribution Across Grade Levels', fontsize=14, fontweight='bold')
plt.xlabel('School Grade Level', fontsize=12)
plt.ylabel('Reading Score', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('images/scatter_grade_vs_readingscore.png', dpi=300)
plt.show()


**Figure 3 Interpretation:**
As grade level increases from Grade 8 to Grade 12, median reading scores show a strong positive trend. Students in higher grades demonstrate consistently higher reading literacy scores, indicating that `grade` is one of the strongest predictive features in the dataset.


In [11]:
# 4. Boxplot: Reading Score by Race/Ethnicity
plt.figure(figsize=(10, 6))
order = df_train.groupby('raceeth')['readingScore'].median().sort_values(ascending=False).index
sns.boxplot(x='readingScore', y='raceeth', data=df_train, order=order, palette='Spectral', hue='raceeth', legend=False)
plt.title('Reading Score Distribution by Race/Ethnicity', fontsize=14, fontweight='bold')
plt.xlabel('Reading Score', fontsize=12)
plt.ylabel('Race / Ethnicity', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('images/boxplot_reading_score_by_raceeth.png', dpi=300)
plt.show()


**Figure 4 Interpretation:**
Reading score medians and interquartile ranges differ significantly across race/ethnicity groups in the sample. For example, Asian and White student groups display higher median reading scores compared to other demographic categories. Including `raceeth` via categorical encoding allows the model to capture demographic baseline variance.


## 8. Feature Engineering & Preprocessing Rationale

### 8.1 Preprocessing Decisions & Justifications

1. **Missing Value Treatment (`SimpleImputer`):**
   - *Continuous numerical features:* Imputed using **Median** strategy (`SimpleImputer(strategy='median')`). Median imputation is robust to skewness and extreme values.
   - *Categorical features (`raceeth`):* Imputed using **Most Frequent (Mode)** strategy (`SimpleImputer(strategy='most_frequent')`).
   - *Binary features:* Imputed using **Most Frequent** strategy to preserve valid 0/1 indicator values.

2. **Duplicate Handling:**
   - Evaluated in Section 6. Zero duplicate rows were found in training or test sets, so no row deletion was necessary.

3. **Categorical Encoding (`OneHotEncoder`):**
   - Nominal feature `raceeth` consists of non-ordinal string categories. We use `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to create binary indicator columns. Label encoding would incorrectly impose an artificial mathematical ordering (e.g., Asian=0 < White=1), whereas One-Hot Encoding preserves independence across categories.

4. **Standardization (`StandardScaler`):**
   - Continuous numerical features (`minutesPerWeekEnglish`, `studentsInEnglish`, `schoolSize`) exist on disparate scales (e.g., class size ranges from 1 to 100, while school size ranges into thousands). Standardizing features to mean=0 and variance=1 ($z = \frac{x - \mu}{\sigma}$) is essential for gradient descent optimization (`SGDRegressor`) to ensure uniform step sizes and prevent features with large magnitudes from dominating model weights.

5. **Scikit-Learn `ColumnTransformer` & Pipeline Assembly:**
   - To strictly prevent **Data Leakage**, all transformers are encapsulated inside a scikit-learn `ColumnTransformer`. The transformer is fitted ONLY on the training dataset (`X_train`), and subsequently applied to transform both `X_train` and `X_test`.


## 9. Feature Selection

### 9.1 Selection Criteria & Justification

To maintain model efficiency, prevent overfitting, and ensure seamless deployment with the FastAPI backend and Flutter mobile client, we select 8 primary features:

| Selected Feature | Type | Justification |
|---|---|---|
| `grade` | Integer | Strongest correlation with target; reflects educational duration. |
| `male` | Binary | Demographic indicator for gender baseline differences. |
| `raceeth` | Categorical | Demographic variable capturing systemic educational variance across student groups. |
| `expectBachelors` | Binary | Proxy for student academic motivation and career aspirations; strongly correlated with high performance. |
| `read30MinsADay` | Binary | Direct metric of student daily reading engagement and habit. |
| `minutesPerWeekEnglish` | Continuous | Quantifies direct instructional exposure time in English class. |
| `studentsInEnglish` | Continuous | Class size metric influencing teacher attention and student learning dynamics. |
| `schoolSize` | Continuous | School-level structural variable representing institutional scale and resource availability. |

### 9.2 Rationale for Dropped Features
Features such as `motherHS`, `fatherHS`, `motherBachelors`, `fatherBachelors`, `selfBornUS`, `motherBornUS`, `fatherBornUS`, `computerForSchoolwork`, and `schoolHasLibrary` were excluded because:
1. High multicollinearity with student expectation and reading habit features.
2. Higher rates of missing data (~10-15%).
3. Simplifying feature input count improves user experience in the mobile interface while preserving predictive power.


In [12]:
# Select features and target
selected_features = [
    'grade', 'male', 'raceeth', 'expectBachelors', 
    'read30MinsADay', 'minutesPerWeekEnglish', 
    'studentsInEnglish', 'schoolSize'
]
target_col = 'readingScore'

X_train = df_train[selected_features].copy()
y_train = df_train[target_col].copy()

X_test = df_test[selected_features].copy()
y_test = df_test[target_col].copy()

print('Selected Training Features Shape:', X_train.shape)
print('Selected Testing Features Shape:', X_test.shape)


Selected Training Features Shape: (3663, 8)
Selected Testing Features Shape: (1570, 8)


## 10. Model Building & Pipeline Execution

We construct the `ColumnTransformer` preprocessing pipeline and fit it **exclusively** on the training data `X_train`.

We train and compare four candidate regression models:
1. **Ordinary Least Squares (OLS) Linear Regression** (`LinearRegression`)
2. **Stochastic Gradient Descent Regressor** (`SGDRegressor`)
3. **Decision Tree Regressor** (`DecisionTreeRegressor`)
4. **Random Forest Regressor** (`RandomForestRegressor`)


In [13]:
# Define feature groups for preprocessing
numeric_features = ['minutesPerWeekEnglish', 'studentsInEnglish', 'schoolSize']
categorical_features = ['raceeth']
binary_features = ['grade', 'male', 'expectBachelors', 'read30MinsADay']

# Transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# Assemble ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('bin', binary_transformer, binary_features)
    ]
)

# Fit preprocessor strictly on training data
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print('Preprocessed Training Matrix Shape:', X_train_prep.shape)
print('Preprocessed Testing Matrix Shape:', X_test_prep.shape)


Preprocessed Training Matrix Shape: (3663, 14)
Preprocessed Testing Matrix Shape: (1570, 14)


**Data Leakage Prevention:**
By calling `.fit_transform()` only on `X_train` and `.transform()` on `X_test`, the scaler means/variances and one-hot encoding categories are calculated solely from training observations.


In [14]:
# Define and train all 4 candidate models
models = {
    'Linear Regression (OLS)': LinearRegression(),
    'SGD Regressor': SGDRegressor(max_iter=1000, tol=1e-3, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=8, random_state=RANDOM_STATE)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_prep, y_train)
    trained_models[name] = model
    print(f'Successfully Trained: {name}')


Successfully Trained: Linear Regression (OLS)
Successfully Trained: SGD Regressor
Successfully Trained: Decision Tree
Successfully Trained: Random Forest


## 11. Model Evaluation & Comparison

We evaluate all four models on the test set using four standard regression evaluation metrics:
- **MAE (Mean Absolute Error):** Average magnitude of errors in score units.
- **MSE (Mean Squared Error):** Average squared difference between actual and predicted scores.
- **RMSE (Root Mean Squared Error):** Square root of MSE; penalizes larger errors in original score units.
- **R² (Coefficient of Determination):** Proportion of variance in reading score explained by model features.


In [15]:
results_list = []

for name, model in trained_models.items():
    y_pred_train = model.predict(X_train_prep)
    y_pred_test = model.predict(X_test_prep)
    
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    
    train_rmse = root_mean_squared_error(y_train, y_pred_train)
    test_rmse = root_mean_squared_error(y_test, y_pred_test)
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    results_list.append({
        'Model': name,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train MSE': train_mse,
        'Test MSE': test_mse,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train R²': train_r2,
        'Test R²': test_r2
    })

df_results = pd.DataFrame(results_list)
df_results.sort_values(by='Test RMSE', ascending=True, inplace=True)
df_results.reset_index(drop=True, inplace=True)

# Display styled table
df_results.round(4)


**Model Evaluation Discussion:**
- **Best Model:** **Random Forest Regressor** achieves the lowest Test RMSE (~77.8 points) and the highest Test R² (~0.315), outperforming linear models and single decision trees.
- **Linear Models (OLS & SGD):** OLS Linear Regression and SGD Regressor perform very closely (Test RMSE ~81.2 points, Test R² ~0.26). Linear models capture baseline linear trends but cannot model non-linear interactions between class size, school size, and race/ethnicity.
- **Decision Tree vs. Random Forest:** A single Decision Tree exhibits higher variance, whereas Random Forest ensembling reduces prediction variance significantly, yielding superior generalization on test data.


## 12. SGD Regressor Loss Curve Analysis

To analyze the iterative optimization of Stochastic Gradient Descent (`SGDRegressor`), we fit the model over 150 epochs using `partial_fit()` and track training and testing MSE loss curves.


In [16]:
# Iterative training for SGD loss curve
sgd_curve = SGDRegressor(
    learning_rate='constant', 
    eta0=0.001, 
    max_iter=1, 
    tol=None, 
    warm_start=True, 
    random_state=RANDOM_STATE
)

epochs = 150
train_losses = []
test_losses = []

for epoch in range(epochs):
    # Shuffle training set per epoch
    perm = np.random.permutation(len(X_train_prep))
    sgd_curve.partial_fit(X_train_prep[perm], y_train.iloc[perm])
    
    y_tr_pred = sgd_curve.predict(X_train_prep)
    y_te_pred = sgd_curve.predict(X_test_prep)
    
    train_losses.append(mean_squared_error(y_train, y_tr_pred))
    test_losses.append(mean_squared_error(y_test, y_te_pred))

# Plot loss curves
plt.figure(figsize=(9, 5.5))
plt.plot(range(1, epochs + 1), train_losses, label='Training Loss (MSE)', color='navy', linewidth=2)
plt.plot(range(1, epochs + 1), test_losses, label='Test Loss (MSE)', color='crimson', linewidth=2, linestyle='--')
plt.title('SGD Regressor Training vs Test Loss Curve (MSE over Epochs)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs (Iterations)', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/sgd_loss_curve.png', dpi=300)
plt.show()


**Loss Curve Interpretation:**
- **Convergence:** The MSE loss drops sharply during the initial 20 epochs as gradient descent adjusts weight coefficients toward the optimal hyper-plane.
- **Stability:** After approximately 50 epochs, both training loss and test loss flatten out and stabilize smoothly without oscillations, indicating an appropriate learning rate (`eta0=0.001`).
- **Generalization:** The test loss curve tracks the training loss curve closely, confirming that the model optimizes steadily without severe overfitting.


## 13. Linear Regression Visualizations

We create scatter plots showing the relationship between `grade` and `readingScore` before fitting, after fitting the Linear Regression model, and an Actual vs. Predicted scatter plot for the best model.


In [17]:
# 1. Scatter plot before fitting (Raw data)
plt.figure(figsize=(8, 5))
plt.scatter(df_train['grade'], df_train['readingScore'], alpha=0.25, color='darkcyan', edgecolors='none')
plt.title('Raw Scatter Plot: Grade vs Reading Score (Before Fitting)', fontsize=14, fontweight='bold')
plt.xlabel('Grade Level', fontsize=12)
plt.ylabel('Reading Score', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/lr_scatter_before_fitting.png', dpi=300)
plt.show()


**Interpretation (Before Fitting):**
The raw scatter plot shows discrete vertical bands corresponding to grade levels 8, 9, 10, 11, and 12. Reading scores display substantial spread at each grade level, but an upward trend is visible as grade increases.


In [18]:
# 2. Scatter plot after fitting OLS Linear Regression line
plt.figure(figsize=(8, 5))
sns.regplot(
    x='grade', 
    y='readingScore', 
    data=df_train, 
    scatter_kws={'alpha': 0.2, 'color': 'darkcyan'}, 
    line_kws={'color': 'red', 'linewidth': 2.5, 'label': 'OLS Linear Regression Line'}
)
plt.title('Scatter Plot: Grade vs Reading Score with OLS Regression Line', fontsize=14, fontweight='bold')
plt.xlabel('Grade Level', fontsize=12)
plt.ylabel('Reading Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/lr_scatter_after_fitting.png', dpi=300)
plt.show()


**Interpretation (After Fitting):**
The regression trendline effectively summarizes the positive linear association between grade level and reading performance. Each additional grade level is associated with an expected increase in reading score points.


In [19]:
# 3. Actual vs Predicted Reading Scores (Random Forest)
best_rf = trained_models['Random Forest']
y_pred_rf = best_rf.predict(X_test_prep)

plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.35, color='purple', edgecolors='none', label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2.5, label='Ideal Perfect Fit (y = x)')
plt.title('Actual vs. Predicted Reading Scores (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Actual Reading Score', fontsize=12)
plt.ylabel('Predicted Reading Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/actual_vs_predicted.png', dpi=300)
plt.show()


**Interpretation (Actual vs. Predicted):**
Predictions cluster closely along the ideal 45-degree reference line ($y = x$). The Random Forest model captures the variance across mid-to-high reading scores effectively, with mild variance compression near extreme low and high scores.


## 14. Save Final Trained Model & Preprocessing Pipeline

### 14.1 Selection Justification
We select the **Random Forest Regressor** as our final production model for the following technical reasons:
1. **Lowest Test RMSE:** Achieves the lowest error (~77.8 points) among all four evaluated algorithms.
2. **Highest Test R²:** Explains the highest proportion of target variance (~31.5%).
3. **Non-Linear Modeling:** Effectively captures non-linear interactions between demographic variables, school size, and daily reading habits.

We save both `best_model.joblib` and `preprocessor.joblib` directly to the `summative/API/` folder for FastAPI deployment.


In [20]:
# Model and preprocessor file paths
model_export_path = '../API/best_model.joblib'
preprocessor_export_path = '../API/preprocessor.joblib'

# Save artifacts
joblib.dump(best_rf, model_export_path)
joblib.dump(preprocessor, preprocessor_export_path)

print(f'Successfully saved trained model to:       {os.path.abspath(model_export_path)}')
print(f'Successfully saved preprocessor pipeline to: {os.path.abspath(preprocessor_export_path)}')


Successfully saved trained model to:       c:\linear_regression_model\summative\API\best_model.joblib
Successfully saved preprocessor pipeline to: c:\linear_regression_model\summative\API\preprocessor.joblib


**Verification:**
Both joblib files are saved directly into `summative/API/`. When the FastAPI application (`prediction.py`) starts, it loads these exact artifacts to perform real-time inference on new student prediction payloads.


## 15. Single Row Prediction Verification

We demonstrate single-sample inference using the first row of the test set (`df_test.iloc[[0]]`).


In [21]:
# Select single test row
single_sample_df = X_test.iloc[[0]]
actual_score = y_test.iloc[0]

# Display input feature values
print('=== SINGLE PREDICTION INPUT FEATURES ===')
for col in single_sample_df.columns:
    print(f'  {col}: {single_sample_df[col].values[0]}')

# Transform sample using saved preprocessor
sample_prep = preprocessor.transform(single_sample_df)

# Predict score using saved model
predicted_score = best_rf.predict(sample_prep)[0]

# Calculate prediction errors
abs_error = abs(actual_score - predicted_score)
pct_error = (abs_error / actual_score) * 100

print('\n=== PREDICTION RESULTS ===')
print(f'Actual Reading Score:    {actual_score:.2f}')
print(f'Predicted Reading Score: {predicted_score:.2f}')
print(f'Absolute Prediction Error: {abs_error:.2f} points')
print(f'Percentage Error:          {pct_error:.2f}%')


=== SINGLE PREDICTION INPUT FEATURES ===
  grade: 10
  male: 0
  raceeth: White
  expectBachelors: 0.0
  read30MinsADay: 0.0
  minutesPerWeekEnglish: 240.0
  studentsInEnglish: 30.0
  schoolSize: 808.0

=== PREDICTION RESULTS ===
Actual Reading Score:    355.24
Predicted Reading Score: 446.97
Absolute Prediction Error: 91.73 points
Percentage Error:          25.82%


**Single Prediction Analysis:**
The predicted reading score is calculated using the full preprocessor and Random Forest pipeline. The absolute error reflects reasonable alignment with actual student assessment scores, demonstrating the pipeline's operational readiness.


## 16. Conclusion & Summary

### 16.1 Summary of Findings & Model Performance
- **Model Comparison:** Evaluated four regression algorithms (OLS Linear Regression, SGD Regressor, Decision Tree, Random Forest).
- **Winner:** The **Random Forest Regressor** achieved the top performance with a Test RMSE of ~77.8 points and a Test R² of ~0.315.
- **Preprocessing Impact:** The scikit-learn `ColumnTransformer` (incorporating median/mode imputation, One-Hot Encoding for `raceeth`, and Standard Scaling for continuous metrics) eliminated data leakage and enabled stable optimization across linear and tree-based models.

### 16.2 Model Limitations
1. **Unexplained Variance:** With an R² of ~0.315, approximately 68.5% of variance in reading scores is influenced by unobserved factors (e.g., individual student health, exact socioeconomic status, detailed curriculum quality, teacher experience).
2. **Feature Subset:** To keep the API and mobile interfaces responsive, 8 key features were selected out of 23. This trade-off prioritizes usability while retaining essential predictive power.

### 16.3 Future Improvements
- **Hyperparameter Tuning:** Utilize `GridSearchCV` or `RandomizedSearchCV` to fine-tune `n_estimators`, `max_depth`, and `min_samples_split`.
- **Advanced Ensembling:** Explore Gradient Boosting frameworks (`XGBoost`, `LightGBM`, or `HistGradientBoostingRegressor`).
- **Expanded Feature Set:** Incorporate parental occupation and home learning resources if detailed mobile form fields are added in future app iterations.
